In [1]:
import torch
import hydra
import train
import pathlib
import src.utils.rebasin.activation_matching
import numpy as np
import src.utils.interpolation
device="cuda:0"

checkpoint_directories = {
    "mlp_symmetry0": "outputs/2025-12-17/12-18-16_mlp_mnist_sym-0__3pb1rw8b__kk13eg0q"
}
checkpoint_path_1, checkpoint_path_2 = [checkpoint_directories["mlp_symmetry0"]+f"/checkpoint_epoch_100_model_{i}.pt" for i in [4,5]]

with hydra.initialize(version_base=None, config_path=str(pathlib.Path(checkpoint_path_1).parent)):
    cfg = hydra.compose(config_name="config")
cfg.dataset.batch_size = 1000
data_info = train.setup_data_loaders(cfg)

model1 = train.create_model(cfg, mask_seed=-1).to(device)
model1_permuted = train.create_model(cfg, mask_seed=-1).to(device)
model2 = train.create_model(cfg, mask_seed=-1).to(device)
model2_permuted = train.create_model(cfg, mask_seed=-1).to(device)
model1.load_state_dict(torch.load(checkpoint_path_1, map_location=device)["model_state_dict"])
model2.load_state_dict(torch.load(checkpoint_path_2, map_location=device)["model_state_dict"])

<All keys matched successfully>

In [3]:
import copy

permutation_spec = src.utils.rebasin.activation_matching.mlp_permutation_spec(3, norm=True, bias=True)
output_permutation_by_layer = src.utils.rebasin.activation_matching.activation_matching(
    permutation_spec, model1, model2, data_info["train_loader"], device=device
)
permuted_params_1, permuted_params_2 = [src.utils.rebasin.activation_matching.apply_permutation(
    permutation_spec,
    {"P_0": output_permutation_by_layer["lins.1.input"].optimal_permutation,
     "P_1": output_permutation_by_layer["lins.2.input"].optimal_permutation,
     "P_2": output_permutation_by_layer["lins.3.input"].optimal_permutation},
    copy.deepcopy(m).state_dict()
) for m in [model1, model2]]
model1_permuted.load_state_dict(permuted_params_1)
model2_permuted.load_state_dict(permuted_params_2)

<All keys matched successfully>

In [4]:
import copy

# Sanity check: check LMC
src.utils.interpolation.interpolate_models(copy.deepcopy(model1), model1.state_dict(), model2.state_dict(), data_info["train_loader"], data_info["val_loader"], data_info["test_loader"], steps=10, device="cuda:0")

src.utils.interpolation.interpolate_models(copy.deepcopy(model1), permuted_params_1, model2.state_dict(), data_info["train_loader"], data_info["val_loader"], data_info["test_loader"], steps=10, device="cuda:0");
src.utils.interpolation.interpolate_models(copy.deepcopy(model1), model1.state_dict(), permuted_params_2, data_info["train_loader"], data_info["val_loader"], data_info["test_loader"], steps=10, device="cuda:0");

Step  1/11 (λ=0.000): Train Acc=100.00%, Val Acc=98.75%, Test Acc=98.77%, Train Loss=0.0000, Val Loss=0.0767, Test Loss=0.0885
Step  2/11 (λ=0.100): Train Acc=100.00%, Val Acc=98.70%, Test Acc=98.70%, Train Loss=0.0000, Val Loss=0.0668, Test Loss=0.0780
Step  3/11 (λ=0.200): Train Acc=99.99%, Val Acc=98.48%, Test Acc=98.50%, Train Loss=0.0004, Val Loss=0.0629, Test Loss=0.0721
Step  4/11 (λ=0.300): Train Acc=99.80%, Val Acc=98.08%, Test Acc=97.96%, Train Loss=0.0071, Val Loss=0.0719, Test Loss=0.0788
Step  5/11 (λ=0.400): Train Acc=97.89%, Val Acc=96.58%, Test Acc=96.15%, Train Loss=0.0690, Val Loss=0.1247, Test Loss=0.1314
Step  6/11 (λ=0.500): Train Acc=94.65%, Val Acc=93.43%, Test Acc=93.46%, Train Loss=0.2056, Val Loss=0.2549, Test Loss=0.2397
Step  7/11 (λ=0.600): Train Acc=98.49%, Val Acc=96.70%, Test Acc=96.96%, Train Loss=0.0641, Val Loss=0.1264, Test Loss=0.1136
Step  8/11 (λ=0.700): Train Acc=99.59%, Val Acc=97.65%, Test Acc=97.92%, Train Loss=0.0160, Val Loss=0.0859, Test Lo

In [7]:
import torch
import torch.nn as nn
from torch.func import functional_call, jvp, vjp

# General version with proper Hessian handling
def ggn_quadratic_form_general(model, loss_fn, params_ref, params_1, params_2, inputs, targets):
    v = {k: params_1[k] - params_2[k] for k in params_1.keys()}
    
    # Step 1: Compute Jv using forward-mode
    def residual_fn(p):
        return functional_call(model, p, inputs)
    
    residuals_ref, jv = jvp(residual_fn, (params_ref,), (v,))
    
    # Step 2: Compute Hessian-vector product using backward-mode
    # Make residuals require grad for Hessian computation
    residuals_ref_grad = residuals_ref.detach().requires_grad_(True)
    loss_val = loss_fn(residuals_ref_grad, targets)
    
    # Compute gradient of loss w.r.t. residuals
    grad_r, = torch.autograd.grad(loss_val, residuals_ref_grad, create_graph=True)
    
    # Compute HVP: gradient of (grad_r · jv) w.r.t. residuals
    hvp, = torch.autograd.grad(
        grad_r, residuals_ref_grad, grad_outputs=jv,
        retain_graph=False
    )
    
    # Step 3: Compute (Jv)ᵀ H (Jv)
    return torch.sum(jv * hvp, dim=1)

In [5]:
input0, target0 = next(iter(data_info["train_loader"])); input0 = input0.to(device); target0 = target0.to(device)

In [9]:
params_mid_aligned = {k: (dict(model1_permuted.named_parameters())[k]*0.5 + dict(model2.named_parameters())[k]*0.5).detach().clone() for k, v in model1.named_parameters()}
params_mid_unaligned = {k: (dict(model1.named_parameters())[k]*0.5 + dict(model2.named_parameters())[k]*0.5).detach().clone() for k, v in model1.named_parameters()}
params_1 = {k: v.detach().clone() 
            for k, v in model1.named_parameters()}
params_1_permuted = {k: v.detach().clone()
            for k, v in model1_permuted.named_parameters()}
params_2 = {k: v.detach().clone() 
            for k, v in model2.named_parameters()}

ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(),
                           params_ref=params_2,
                           params_1=params_1,
                           params_2=params_2,
                           inputs=input0, targets=target0).mean()

tensor(6.5416e-10, device='cuda:0')

In [10]:
ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(),
                           params_ref=params_2,
                           params_1=params_1_permuted,
                           params_2=params_2,
                           inputs=input0, targets=target0).mean()

tensor(3.8256e-10, device='cuda:0')

In [17]:
ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(), params_1, params_2, params_ref, input0, target0)

In [18]:
ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(), params_1, params_1, params_2, input0, target0)

In [20]:
ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(), params_1, params_1, params_2, input0, target0)

In [21]:
ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(), params_2, params_1, params_2, input0, target0)

In [22]:
ggn_quadratic_form_general(model1, torch.nn.CrossEntropyLoss(), params_2, params_1_permuted, params_2, input0, target0)

In [23]:
def ggn_quadratic_form_general(model, loss_fn, params_ref, params_1, params_2, inputs, targets):
    v = {k: params_1[k] - params_2[k] for k in params_1.keys()}
    
    # Step 1: Compute Jv using forward-mode
    def residual_fn(p):
        return functional_call(model, p, inputs)
    
    residuals_ref, jv = jvp(residual_fn, (params_ref,), (v,))
    
    # Step 2: Compute Hessian-vector product using backward-mode
    # Make residuals require grad for Hessian computation
    residuals_ref_grad = residuals_ref.detach().requires_grad_(True)
    loss_val = loss_fn(residuals_ref_grad, targets)
    
    # Compute gradient of loss w.r.t. residuals
    grad_r, = torch.autograd.grad(loss_val, residuals_ref_grad, create_graph=True)
    
    # Compute HVP: gradient of (grad_r · jv) w.r.t. residuals
    hvp, = torch.autograd.grad(
        grad_r, residuals_ref_grad, grad_outputs=jv,
        retain_graph=False
    )
    
    # Step 3: Compute (Jv)ᵀ H (Jv)
    return torch.sum(jv * hvp)
    

In [24]:
jvp?